# 1 - Crear Spark Session

In [6]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("RetailLambdaStreaming")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

spark.version

26/08/07 19:45:13 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


'3.5.1'

# 2 - Verificar conexión con Kafka

In [7]:
kafka_server = "kafka:9092"

print(kafka_server)

kafka:9092


# 3 - Leer orders desde Kafka

In [8]:
orders_raw = (
    spark.readStream
    .format("kafka")
    .option(
        "kafka.bootstrap.servers",
        kafka_server
    )
    .option(
        "subscribe",
        "orders_topic"
    )
    .option(
        "startingOffsets",
        "latest"
    )
    .load()
)

orders_raw.printSchema()

AnalysisException: Failed to find data source: kafka. Please deploy the application as per the deployment section of Structured Streaming + Kafka Integration Guide.

# 4 - Convertir JSON de orders

In [9]:
from pyspark.sql.types import *

orders_schema = StructType([
    StructField(
        "order_id",
        IntegerType()
    ),
    StructField(
        "order_date",
        TimestampType()
    ),
    StructField(
        "order_customer_id",
        IntegerType()
    ),
    StructField(
        "order_status",
        StringType()
    )
])

### Trasformamos

In [ ]:
from pyspark.sql.functions import *


orders_stream = (
    orders_raw
    .selectExpr(
        "CAST(value AS STRING) json"
    )
    .select(
        from_json(
            col("json"),
            orders_schema
        ).alias("data")
    )
    .select("data.*")
)


orders_stream.printSchema()

# 5 - Mostrar streaming de orders

In [10]:
query_orders = (
    orders_stream
    .writeStream
    .format("console")
    .outputMode("append")
    .option(
        "truncate",
        False
    )
    .start()
)

NameError: name 'orders_stream' is not defined

# 6 - Leer order_items

In [11]:
items_raw = (
    spark.readStream
    .format("kafka")
    .option(
        "kafka.bootstrap.servers",
        kafka_server
    )
    .option(
        "subscribe",
        "order_items_topic"
    )
    .option(
        "startingOffsets",
        "latest"
    )
    .load()
)

AnalysisException: Failed to find data source: kafka. Please deploy the application as per the deployment section of Structured Streaming + Kafka Integration Guide.

In [12]:
items_schema = StructType([

    StructField(
        "order_item_id",
        IntegerType()
    ),

    StructField(
        "order_item_order_id",
        IntegerType()
    ),

    StructField(
        "order_item_product_id",
        IntegerType()
    ),

    StructField(
        "order_item_quantity",
        IntegerType()
    ),

    StructField(
        "order_item_subtotal",
        DoubleType()
    ),

    StructField(
        "order_item_product_price",
        DoubleType()
    )
])

In [13]:
items_stream = (
    items_raw
    .selectExpr(
        "CAST(value AS STRING) json"
    )
    .select(
        from_json(
            col("json"),
            items_schema
        ).alias("data")
    )
    .select("data.*")
)


items_stream.printSchema()

NameError: name 'items_raw' is not defined

# 7 - Mostrar items

In [15]:
query_items = (
    items_stream
    .writeStream
    .format("console")
    .outputMode("append")
    .option(
        "truncate",
        False
    )
    .start()
)

NameError: name 'items_stream' is not defined

In [14]:
orders_stream
       |
       JOIN
       |
items_stream
       |
       v
ventas_por_estado
ventas_por_producto
total_ventas

IndentationError: unexpected indent (2969027943.py, line 2)